# 01. Used Car Listing Data Review & Risk Rule Discovery

This notebook performs Exploratory Data Analysis (EDA) on the Craigslist cars/trucks dataset to discover potential risk signals and establish initial data-driven rules for our Used Car Listing Risk Score System.

### 1. Import Libraries
We start by importing our primary libraries: `pandas` for data manipulation, `numpy` for mathematical operations, and `matplotlib` / `seaborn` for plotting.

In [ ]:
# 1. Import libraries
import os
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 2. Load the Dataset
We load the dataset from `data/cardata.csv`. We include path-resolution checks so that the notebook runs without error whether executed from the project root or from inside the `notebooks/` directory.

In [ ]:
# Define dataset path
# Check both current directory and parent directory to avoid notebook working directory errors
dataset_path = 'data/cardata.csv'
if not os.path.exists(dataset_path):
    if os.path.exists('../data/cardata.csv'):
        dataset_path = '../data/cardata.csv'

# Robust check: if the path is a directory (e.g. Kaggle folder extract), look for files inside it
if os.path.isdir(dataset_path):
    files = os.listdir(dataset_path)
    print(f"'{dataset_path}' is a directory containing: {files}")
    if 'vehicles.csv' in files:
        dataset_path = os.path.join(dataset_path, 'vehicles.csv')
    elif len(files) > 0:
        dataset_path = os.path.join(dataset_path, files[0])

print(f"Resolved dataset path: {dataset_path}")

# Load the dataset (you can add nrows=100000 to sample if memory is limited)
try:
    df = pd.read_csv(dataset_path)
    print(f"Successfully loaded dataset with shape: {df.shape}")
except Exception as e:
    print(f"Error loading CSV file: {e}")
    print("Creating a dummy DataFrame for demonstration purposes.")
    df = pd.DataFrame({
        'id': [1, 2, 3, 4, 5],
        'price': [12000, 0, 450000, 3500, 15000],
        'year': [2015, 2018, 2021, 2002, np.nan],
        'odometer': [85000, 5000, 1200000, 250000, 90000],
        'manufacturer': ['ford', 'toyota', 'ferrari', 'honda', 'chevrolet'],
        'model': ['f-150', 'camry', 'f8', 'civic', np.nan],
        'condition': ['good', 'excellent', 'like new', 'fair', 'good'],
        'cylinders': ['8 cylinders', '4 cylinders', '8 cylinders', '4 cylinders', np.nan],
        'fuel': ['gas', 'gas', 'gas', 'gas', 'gas'],
        'title_status': ['clean', 'rebuilt', 'clean', 'salvage', 'clean'],
        'transmission': ['automatic', 'automatic', 'automatic', 'manual', 'automatic'],
        'drive': ['4wd', 'fwd', 'rwd', 'fwd', np.nan],
        'type': ['truck', 'sedan', 'coupe', 'sedan', 'suv'],
        'paint_color': ['black', 'silver', 'red', 'blue', 'white'],
        'VIN': ['1FTFW1EF5FAXXXXXX', np.nan, 'ZFF86AXXXXXX', np.nan, '1G1YY2EX1FAXXXXXX'],
        'description': ['Beautiful truck in excellent condition.', 'Camry for sale', 'Rare supercar, serious buyers only.', 'old car runs well', 'nice SUV'],
        'image_url': ['http://image1.jpg', 'http://image2.jpg', np.nan, 'http://image4.jpg', 'http://image5.jpg'],
        'posting_date': ['2021-04-15T12:00:00-0400', '2021-04-16T12:00:00-0400', '2021-04-17T12:00:00-0400', '2021-04-18T12:00:00-0400', '2021-04-19T12:00:00-0400'],
        'state': ['mi', 'ca', 'ny', 'or', 'tx']
    })
    print(f"Dummy DataFrame shape: {df.shape}")

### 3. Basic Dataset Information
Let's inspect the dimensions, column names, data types, and basic statistics of the dataset.

In [ ]:
# Show dataset dimensions
print("Dataset Shape:")
print(df.shape)
print("-" * 50)

# Show first few rows
print("First 5 Rows:")
display(df.head())
print("-" * 50)

# Show columns list
print("Dataset Columns:")
print(df.columns.tolist())
print("-" * 50)

# Show DataFrame info (types, non-null counts)
print("Dataset Info:")
df.info()
print("-" * 50)

# Show numerical column summaries
print("Statistical Summary (Numerical Columns):")
display(df.describe())

### 4. Check Missing Values
Understanding missingness in the dataset is a key component for assessing the quality and potential risk of listing metadata.

In [ ]:
# Calculate missing value counts and percentages
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

# Combine into a summary DataFrame, sorted from highest to lowest missing values
missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage (%)': missing_percent
}).sort_values(by='Missing Count', ascending=False)

print("Missing Values Summary:")
display(missing_df)

### 5. Column-Specific Analysis
Let's examine some of the most critical columns for car listings if they exist in our dataset.

In [ ]:
# Define critical columns to inspect
important_columns = [
    'price', 'year', 'odometer', 'manufacturer', 'model', 'condition', 
    'cylinders', 'fuel', 'title_status', 'transmission', 'drive', 
    'type', 'paint_color', 'VIN', 'description', 'image_url', 
    'posting_date', 'state'
]

# Print check for each column
print("Critical Columns Status:")
for col in important_columns:
    exists = col in df.columns
    print(f" - {col:15} : {'EXISTS' if exists else 'MISSING'}")

### 6. Exploratory Feature Engineering
Here, we derive features that could serve as signals for finding anomalies or risk factors in the listings.

In [ ]:
# Get current year
current_year = datetime.datetime.now().year

# Create new features safely
if 'year' in df.columns:
    df['car_age'] = current_year - df['year']
    print("Created 'car_age' column.")
else:
    print("Skipped 'car_age' (missing 'year' column).")

if 'description' in df.columns:
    # Fill NA with empty string to avoid errors in length calculation
    df['description_length'] = df['description'].fillna('').astype(str).str.len()
    print("Created 'description_length' column.")
else:
    print("Skipped 'description_length' (missing 'description' column).")

if 'VIN' in df.columns:
    # Check if VIN is non-null and not empty
    df['has_vin'] = df['VIN'].notnull() & (df['VIN'].astype(str).str.strip() != '')
    print("Created 'has_vin' column.")
else:
    print("Skipped 'has_vin' (missing 'VIN' column).")

if 'image_url' in df.columns:
    # Check if image_url is non-null and not empty
    df['has_image'] = df['image_url'].notnull() & (df['image_url'].astype(str).str.strip() != '')
    print("Created 'has_image' column.")
else:
    print("Skipped 'has_image' (missing 'image_url' column).")

### 7. Explore Risk Signals
We investigate anomalous listings, extreme values, or potential red flags based on our feature variables.

In [ ]:
# Define threshold guidelines for risk checks
LOW_PRICE_THRESHOLD = 500
HIGH_PRICE_THRESHOLD = 150000
HIGH_ODOMETER_THRESHOLD = 300000
OLD_AGE_THRESHOLD = 15
LOW_ODOMETER_THRESHOLD = 10000

# 1. Missing VIN
if 'has_vin' in df.columns:
    missing_vin_pct = (~df['has_vin']).mean() * 100
    print(f"Percentage of listings missing VIN: {missing_vin_pct:.2f}%")

# 2. Missing Image
if 'has_image' in df.columns:
    missing_img_pct = (~df['has_image']).mean() * 100
    print(f"Percentage of listings missing Image: {missing_img_pct:.2f}%")

# 3. Very Short Description (less than 30 characters)
if 'description_length' in df.columns:
    short_desc_pct = (df['description_length'] < 30).mean() * 100
    print(f"Percentage of listings with very short description (< 30 chars): {short_desc_pct:.2f}%")

# 4. Title Status
if 'title_status' in df.columns:
    print('\nTitle Status Value Counts:')
    print(df['title_status'].value_counts(dropna=False))
    risky_titles = ['salvage', 'rebuilt', 'parts only', 'missing']
    risky_pct = df['title_status'].isin(risky_titles).mean() * 100
    print(f"Percentage of listings with risky titles (salvage/rebuilt/parts/missing): {risky_pct:.2f}%")

# 5. Extremely Low and High Prices
if 'price' in df.columns:
    extremely_low_price = (df['price'] > 0) & (df['price'] < LOW_PRICE_THRESHOLD)
    extremely_high_price = df['price'] > HIGH_PRICE_THRESHOLD
    zero_price = df['price'] == 0
    
    print('\nPrice Checks:')
    print(f" - Listings with price = $0: {zero_price.mean()*100:.2f}% ({zero_price.sum()} listings)")
    print(f" - Listings with price between $1 and ${LOW_PRICE_THRESHOLD}: {extremely_low_price.mean()*100:.2f}% ({extremely_low_price.sum()} listings)")
    print(f" - Listings with price > ${HIGH_PRICE_THRESHOLD}: {extremely_high_price.mean()*100:.2f}% ({extremely_high_price.sum()} listings)")

# 6. Unusually High Odometer
if 'odometer' in df.columns:
    high_odometer = df['odometer'] > HIGH_ODOMETER_THRESHOLD
    print('\nOdometer Checks:')
    print(f" - Listings with odometer > {HIGH_ODOMETER_THRESHOLD} miles: {high_odometer.mean()*100:.2f}% ({high_odometer.sum()} listings)")

# 7. Old Car with Very Low Odometer (Potential Odometer Rollback/Fraud)
if 'car_age' in df.columns and 'odometer' in df.columns:
    rollback_risk = (df['car_age'] > OLD_AGE_THRESHOLD) & (df['odometer'] > 0) & (df['odometer'] < LOW_ODOMETER_THRESHOLD)
    print('\nRollback Risk Check:')
    print(f" - Old cars (>{OLD_AGE_THRESHOLD} yrs) with very low odometer (<{LOW_ODOMETER_THRESHOLD} miles): {rollback_risk.mean()*100:.2f}% ({rollback_risk.sum()} listings)")

### 8. Visualizations
Let's visualize key columns and missingness patterns.

In [ ]:
# 1. Missing value percentage bar chart
plt.figure(figsize=(12, 6))
missing_plot_data = missing_percent[missing_percent > 0].sort_values(ascending=False)
if not missing_plot_data.empty:
    sns.barplot(x=missing_plot_data.values, y=missing_plot_data.index, palette='viridis')
    plt.title('Percentage of Missing Values per Column')
    plt.xlabel('Percentage Missing (%)')
    plt.ylabel('Columns')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found to visualize.")

# 2. Price distribution (excluding extreme outliers for readability)
if 'price' in df.columns:
    plt.figure(figsize=(10, 5))
    filtered_price = df[(df['price'] >= 500) & (df['price'] <= 60000)]['price']
    if not filtered_price.empty:
        sns.histplot(filtered_price, kde=True, bins=50, color='skyblue')
        plt.title('Car Price Distribution ($500 - $60,000)')
        plt.xlabel('Price ($)')
        plt.ylabel('Frequency')
        plt.show()
    else:
        print("No prices in range $500-$60,000 to display.")

# 3. Odometer distribution (excluding extreme outliers)
if 'odometer' in df.columns:
    plt.figure(figsize=(10, 5))
    filtered_odo = df[(df['odometer'] >= 1000) & (df['odometer'] <= 250000)]['odometer']
    if not filtered_odo.empty:
        sns.histplot(filtered_odo, kde=True, bins=50, color='salmon')
        plt.title('Odometer Mileage Distribution (1,000 - 250,000 miles)')
        plt.xlabel('Odometer (Miles)')
        plt.ylabel('Frequency')
        plt.show()
    else:
        print("No odometer values in range 1,000-250,000 to display.")

# 4. Year distribution
if 'year' in df.columns:
    plt.figure(figsize=(10, 5))
    filtered_year = df[(df['year'] >= 1990) & (df['year'] <= current_year)]['year']
    if not filtered_year.empty:
        sns.histplot(filtered_year, bins=current_year-1990, color='lightgreen', discrete=True)
        plt.title('Car Year Distribution (1990 - Present)')
        plt.xlabel('Year')
        plt.ylabel('Frequency')
        plt.show()
    else:
        print("No year values from 1990 onwards to display.")

# 5. Title status counts
if 'title_status' in df.columns:
    plt.figure(figsize=(10, 5))
    order = df['title_status'].value_counts().index
    sns.countplot(data=df, x='title_status', order=order, palette='Set2')
    plt.title('Count of Listings by Title Status')
    plt.xlabel('Title Status')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()

# 6. Condition counts
if 'condition' in df.columns:
    plt.figure(figsize=(10, 5))
    order = df['condition'].value_counts().index
    sns.countplot(data=df, x='condition', order=order, palette='pastel')
    plt.title('Count of Listings by Condition')
    plt.xlabel('Condition')
    plt.ylabel('Count')
    plt.show()

## Possible Risk Scoring Rules

Based on our initial data review, here are the proposed heuristic rules to assess a listing's risk:

1. **Missing VIN (`has_vin` is False)**:
   - *Risk Signal*: High risk. Legitimate private sellers and dealerships usually publish the VIN. Its absence prevents buyers from carrying out vehicle history reports (CARFAX, etc.).

2. **Missing Image (`has_image` is False)**:
   - *Risk Signal*: Medium risk. Listings without any photos are less engaging, have a higher chance of being automated spam/scams, or might be hiding cosmetic/structural defects.

3. **Salvage/Rebuilt Title Status (`title_status` is salvage, rebuilt, parts only, missing)**:
   - *Risk Signal*: High risk. Vehicles with rebuilt or salvage titles have a history of major accidents, flooding, or structural damage, impacting their safety, reliability, and resale value.

4. **Extremely Low Price (`price` < $500, or = $0)**:
   - *Risk Signal*: High risk. Often used as bait for down-payments, financing listings disguised as the full price, or malicious scams.

5. **Very Short Description (`description` length < 30 characters)**:
   - *Risk Signal*: Medium risk. Extremely brief descriptions suggest low effort or potential duplicate/spam listings.

6. **Old Car with Unusually Low Odometer (`car_age` > 15 years and `odometer` < 10,000 miles)**:
   - *Risk Signal*: High risk. This is a strong potential indicator of odometer rollback fraud (or tampering), unless verified as a rare classic vehicle.